# mp-spawn-workers — worked example 3: Show that nprocs matches world_size and each rank is unique

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `mp-spawn-workers`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

In `mp.spawn`, `nprocs` controls how many processes are started and directly corresponds to `world_size` in `dist.init_process_group`. Each process receives a unique integer `rank` in the range `[0, nprocs)`. Passing `world_size` as an argument (rather than hardcoding it in the worker) makes the code reusable for different numbers of GPUs, and verifying that all ranks are distinct confirms the spawn mechanics worked correctly.

## Worked solution

**Step 1 — pass `world_size` as an arg so the worker knows its group size.** The worker signature is `worker(rank, world_size, ...)`. We pass `world_size=nprocs` through the `args` tuple.

**Step 2 — each worker records its rank in a shared list.** We use a `Manager().list()` so all processes can append to a single collection. After all workers finish, we have all `nprocs` ranks recorded.

**Step 3 — verify uniqueness and completeness.** After `join=True` returns, we check that the collected ranks form the set `{0, 1, ..., nprocs-1}`. This confirms every rank ran and none was duplicated.

**Step 4 — demonstrate with nprocs=3.** We deliberately use three workers (not the typical 2) to show the pattern generalises beyond the common 2-GPU example.

In [ ]:
import sys
import importlib
import multiprocessing
import torch.multiprocessing as mp

RANK_RECORD_SRC = '''
def worker(rank, world_size, rank_list):
    # Simply record rank to the shared list
    rank_list.append(rank)
'''

def check_all_ranks_unique(nprocs: int = 3):
    with open('/tmp/dd_rank_record.py', 'w') as f:
        f.write(RANK_RECORD_SRC)
    if '/tmp' not in sys.path:
        sys.path.insert(0, '/tmp')
    if 'dd_rank_record' in sys.modules:
        mod = importlib.reload(sys.modules['dd_rank_record'])
    else:
        mod = importlib.import_module('dd_rank_record')

    manager = multiprocessing.Manager()
    rank_list = manager.list()

    mp.spawn(mod.worker, args=(nprocs, rank_list), nprocs=nprocs, join=True)

    collected = sorted(rank_list)
    expected = list(range(nprocs))
    print(f"nprocs={nprocs}, collected ranks={collected}, expected={expected}")
    assert collected == expected, f"Rank mismatch: {collected} != {expected}"
    print("All ranks unique and complete!")
    return collected

try:
    check_all_ranks_unique(nprocs=3)
except Exception as e:
    print(f"Spawn not available: {e}")
    print("Expected output: nprocs=3, collected ranks=[0, 1, 2], expected=[0, 1, 2]")